# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [11]:
phase_3_action_plan = {
    "1. Demographic Stagnation": {
        "Finding": "Nationality, International status, and Special Needs show near-zero variance.",
        "Action": "Remove these features to reduce noise and model complexity."
    },
    "2. High-Cardinality Categoricals": {
        "Finding": "Application mode and Occupations have too many categories (outliers in the 'rare' entry methods).",
        "Action": "Use Target Encoding to map these categories to the mean dropout rate per class."
    },
    "3. Financial & Academic Redundancy": {
        "Finding": "Parental qualifications/occupations and 1st/2nd semester credit data are highly correlated.",
        "Action": "Engineer composite features (e.g., 'Financial_Risk_Score') and drop redundant 2nd sem admin columns."
    },
    "4. Skewed Continuous Variables": {
        "Finding": "Age at enrollment is heavily right-skewed; Curricular units have extreme outlier tails (exam participation).",
        "Action": "Log-transform Age; apply RobustScaler to Curricular units to minimize outlier bias."
    },
    "5. The 'Zero' Signal": {
        "Finding": "0.0 grades in Curricular units are not just 'low'—they represent a distinct behavioral state (Dropout spike).",
        "Action": "Create a 'is_zero_sem1' binary flag to preserve this high-importance signal."
    },
    "6. Class Imbalance": {
        "Finding": "The 'Enrolled' status is a significant minority compared to 'Graduate'.",
        "Action": "Implement SMOTE in the modeling phase to prevent bias toward the majority class."
    },
    "7. Data Integrity Verification": {
        "Observation": "Systematic audit confirmed 0 missing values and 0 duplicate records across the entire dataset.",
        "Technical Insight": "The dataset is characterized by 100% completeness and perfect record independence.",
        "Implemented Action": "No corrective measures (Imputation or Deduplication) are required. The pipeline will proceed directly to feature transformation, ensuring 100% signal retention for Phase 3."
    }
}

print("PHASE 2: ACTION PLAN SUMMARY")
for key, content in phase_3_action_plan.items():
    print(f"\n{key.upper()}")
    for k, v in content.items():
        print(f"  > {k}: {v}")

PHASE 2: ACTION PLAN SUMMARY

1. DEMOGRAPHIC STAGNATION
  > Finding: Nationality, International status, and Special Needs show near-zero variance.
  > Action: Remove these features to reduce noise and model complexity.

2. HIGH-CARDINALITY CATEGORICALS
  > Finding: Application mode and Occupations have too many categories (outliers in the 'rare' entry methods).
  > Action: Use Target Encoding to map these categories to the mean dropout rate per class.

3. FINANCIAL & ACADEMIC REDUNDANCY
  > Finding: Parental qualifications/occupations and 1st/2nd semester credit data are highly correlated.
  > Action: Engineer composite features (e.g., 'Financial_Risk_Score') and drop redundant 2nd sem admin columns.

4. SKEWED CONTINUOUS VARIABLES
  > Finding: Age at enrollment is heavily right-skewed; Curricular units have extreme outlier tails (exam participation).
  > Action: Log-transform Age; apply RobustScaler to Curricular units to minimize outlier bias.

5. THE 'ZERO' SIGNAL
  > Finding: 0.0 g

In [12]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import RobustScaler

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [13]:
# Load the dataset from Phase 2
DATA_PATH = r'..\data\raw\datasetOne.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 4424 rows x 35 columns


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,10,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,4,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,10,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,4,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,10,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [14]:

df_clean = df.copy()

# Select Relevant Columns
columns_to_keep = [
    'Marital status', 'Application mode', 'Application order', 'Course', 
    'Daytime/evening attendance', 'Previous qualification', 'Displaced', 
    'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 
    'Age at enrollment', 'Curricular units 1st sem (enrolled)', 
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 
    'Curricular units 1st sem (grade)', 'Target', 'Curricular units 2nd sem (grade)', 
    'Curricular units 2nd sem (approved)',
]

# 3. Document Rationale for Exclusions
columns_to_drop = [
    'Nationality', 'International', 'Educational special needs', # Zero/Near-zero variance
    'Curricular units 1st sem (without evaluations)',            # Zero variance
    'Curricular units 2nd sem (without evaluations)',            # Zero variance
    'Unemployment rate', 'Inflation rate', 'GDP',                 # Noise (Zero correlation to Target)
    'Curricular units 2nd sem (enrolled)',                        # Redundant (0.94 corr with Sem 1)
    'Curricular units 2nd sem (credited)'                         # Redundant (0.94 corr with Sem 1)
]

drop_reason = {
    "Demographics": {
        "Insight": "Nationality, International, and Special Needs showed near-zero variance; providing no signal.",
        "Evidence_Graph": "Count Plots (Bar Charts) — Showed 98%+ of students belonged to a single category, meaning the model can't 'learn' anything from the rare cases."
    },
    "Macro-Economics": {
        "Insight": "Unemployment, Inflation, and GDP showed no statistical correlation with academic success in EDA.",
        "Evidence_Graph": "Correlation Heatmap — These features had coefficients near 0.0 with the 'Target' variable, indicating they are just 'noise' in this specific dataset."
    },
    "Admin Redundancy": {
        "Insight": "2nd Sem Enrolled/Credited are >0.90 correlated with 1st Sem; keeping them causes multicollinearity.",
        "Evidence_Graph": "Correlation Matrix & Scatter Plots — Showed a near-perfect diagonal line between Sem 1 and Sem 2 admin counts. Keeping both confuses the model's weight assignment."
    },
    "Zero Variance": {
        "Insight": "Units 'without evaluations' were 0 for almost the entire dataset.",
        "Evidence_Graph": "Histograms — Showed a single massive spike at 0.0 with no spread (standard deviation), making the column mathematically useless for splitting data."
    }
}

# Apply the selection
df_clean = df_clean[columns_to_keep].copy()

# Note: Since we only included 'columns_to_keep', the 'columns_to_drop' are effectively already excluded. 

print(f"Original Dataset Shape: {df.shape}")
print(f"Final Selection Shape (Binary + Pruned): {df_clean.shape}")

# Verify current columns
print("\nActive Features for Phase 3:")
print(df_clean.columns.tolist())

Original Dataset Shape: (4424, 35)
Final Selection Shape (Binary + Pruned): (4424, 19)

Active Features for Phase 3:
['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Displaced', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Target', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (approved)']


In [15]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [16]:
# Handle missing values (Safety Check)
# Since the 'Missing Values Report' showed 0 nulls, no imputation is strictly necessary.

df_clean = df_clean.copy()

missing_count = df_clean.isnull().sum().sum()
if missing_count == 0:
    print("Dataset is already clean. No missing values detected.")
else:
    numerical_cols = df_clean.select_dtypes(include=np.number).columns
    df_clean[numerical_cols] = df_clean[numerical_cols].fillna(df_clean[numerical_cols].median())
    print(f"Handled {missing_count} unexpected missing values.")


Dataset is already clean. No missing values detected.


In [17]:
# TODO: Remove duplicate records.

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows.")
print(f"Remaining: {after} rows.")

Removed 33 duplicate rows.
Remaining: 4391 rows.


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [18]:

# --- 1. Derived Attributes (Signal Enhancement) ---
# Academic Efficiency: Ratio of passing to effort
df_clean['Academic_Efficiency'] = df_clean['Curricular units 1st sem (approved)'] / (df_clean['Curricular units 1st sem (evaluations)'] + 1e-9)

# Financial Risk Score: Combined scale (Higher = More Risk)
# Note: We invert 'Tuition fees' because 1=Paid, 0=Debt. 
df_clean['Financial_Risk_Score'] = df_clean['Debtor'] + (1 - df_clean['Tuition fees up to date'])

# Maturity-Entry Interaction: Capturing the non-traditional risk profile
# Application Mode 18 = 'Over 23 years old'
df_clean['is_mature_entry'] = ((df['Age at enrollment'] > 25) & (df['Application mode'].isin([8, 39]))).astype(int)

# --- 2. Binning / Binary Collapse ---
# Zero-Failure Flag: Captures the 0.0 grade spike from our histograms
df_clean['is_zero_sem1'] = (df_clean['Curricular units 1st sem (grade)'] == 0).astype(int)

# Marital Status: Single (1) vs. All others (Riskier)
df_clean['is_single'] = (df_clean['Marital status'] == 1).astype(int)

# Previous Qualification: Standard Secondary (1) vs. Others
df_clean['is_standard_secondary'] = (df_clean['Previous qualification'] == 1).astype(int)

# Credited Units: 0 vs. Any
df_clean['has_credits'] = (df['Curricular units 1st sem (credited)'] > 0).astype(int)

# Academic Momentum: Total progress in the first year
df_clean['Academic_Momentum'] = df['Curricular units 1st sem (approved)'] + df['Curricular units 2nd sem (approved)']

# Late Switcher Flag: Change of Course after 22 years old likely indicates a non-traditional student with higher dropout risk
df_clean['late_switcher'] = ((df['Age at enrollment'] > 22) & (df['Application mode'] == 15)).astype(int)

# ----------- Grade Velocity & Recovery Signal -------------------------------------------------------------------------------
# Raw Velocity: Performance change (Negative = Performance Decay)
df_clean['Grade_Velocity'] = df['Curricular units 2nd sem (grade)'] - df['Curricular units 1st sem (grade)']

# Recovery Signal: Specifically flags students who failed Sem 1 but passed Sem 2
# This helps the model distinguish between 'Consistent' and 'Improving' students
df_clean['Recovery_Signal'] = df_clean['Grade_Velocity'] * df_clean['is_zero_sem1']


In [19]:
# --- 3. Scaling & Normalization ---
# RobustScaler handles the outliers we capped/log-transformed earlier
scaler = RobustScaler()
numerical_to_scale = [
    'Curricular units 1st sem (enrolled)', 
    'Curricular units 1st sem (evaluations)', 
    'Curricular units 1st sem (approved)', 
    'Curricular units 1st sem (grade)',
    'Academic_Efficiency',
    'Academic_Momentum',
    'Grade_Velocity',
    'Recovery_Signal'
]

df_clean[numerical_to_scale] = scaler.fit_transform(df_clean[numerical_to_scale])
print("Task 3 Complete: Features Constructed and Scaled.")

Task 3 Complete: Features Constructed and Scaled.


In [20]:
# Target Encoding 
# Convert complex categories into a single numeric "Risk Score" based on the probability of dropping out.

# Temporary mapping for risk calculation: 
# Dropout (High Risk) = 1, Enrolled (Medium) = 0.5, Graduate (Low) = 0
risk_mapping = {'Dropout': 1.0, 'Enrolled': 0.5, 'Graduate': 0.0}
temp_target_for_risk = df['Target'].map(risk_mapping)

high_card_features = ["Mother's occupation", "Father's occupation", "Application mode"]
scaler = RobustScaler()

for col in high_card_features:
    if col in df.columns:
        # Calculate the mean 'Risk' per category
        encoding_map = temp_target_for_risk.groupby(df[col]).mean()
        
        # Create the new risk feature name
        new_col_name = f"{col.replace(' ', '_')}_risk"
        
        # Map risk scores and fill rare categories with the global mean risk
        df_clean[new_col_name] = df[col].map(encoding_map).fillna(temp_target_for_risk.mean())
        
        # Scale the new feature to match our normalized ranges
        df_clean[new_col_name] = scaler.fit_transform(df_clean[[new_col_name]])

In [33]:
# Run this for each high_card_feature (Mothers_occupation, Fathers_occupation, Application_mode)
# these values are used to create the new risk features in the loop above, 
# but here we just print the mappings for transparency and documentation & deployment purposes
mapping = temp_target_for_risk.groupby(df["Mother's occupation"]).mean().to_dict()
print("mothers occupation mappings:")
print(mapping)
mapping2 = temp_target_for_risk.groupby(df["Father's occupation"]).mean().to_dict()
print("fathers occupation mappings:")
print(mapping2)
mapping3 = temp_target_for_risk.groupby(df["Application mode"]).mean().to_dict()
print("application mode mappings:")
print(mapping3)

mothers occupation mappings:
{1: 0.6909722222222222, 2: 0.45588235294117646, 3: 0.44339622641509435, 4: 0.3831908831908832, 5: 0.3935128518971848, 6: 0.38301886792452833, 7: 0.3626373626373626, 8: 0.38235294117647056, 9: 0.5138888888888888, 10: 0.3944197844007609, 11: 0.5, 12: 0.7285714285714285, 13: 0.8235294117647058, 14: 0.5, 15: 0.42857142857142855, 16: 0.0, 17: 0.5, 18: 0.3333333333333333, 19: 0.5, 20: 0.25, 21: 0.3333333333333333, 22: 0.16666666666666666, 23: 0.16666666666666666, 24: 0.25, 25: 0.0, 26: 0.0, 27: 0.5, 28: 0.4, 29: 0.21153846153846154, 30: 0.3, 31: 0.25, 32: 0.4090909090909091}
fathers occupation mappings:
{1: 0.65234375, 2: 0.44402985074626866, 3: 0.48223350253807107, 4: 0.37890625, 5: 0.45595854922279794, 6: 0.3905038759689923, 7: 0.384297520661157, 8: 0.35960960960960964, 9: 0.38207547169811323, 10: 0.4004950495049505, 11: 0.42105263157894735, 12: 0.7076923076923077, 13: 0.7368421052631579, 14: 0.0, 15: 0.25, 16: 0.375, 17: 0.75, 18: 0.0, 19: 0.5, 20: 0.25, 21: 0

In [ ]:
# Label Encoding for the Actual Target
# Convert the 'Target' strings into integers for the model to predict.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_clean['Target'] = le.fit_transform(df_clean['Target'])

print("\nFinal Categorical Encoding Complete.")
print("Target Mapping Key")
for index, label in enumerate(le.classes_):
    print(f"Class {index} -> {label}")


Final Categorical Encoding Complete.
Target Mapping Key
Class 0 -> 0.0
Class 1 -> 1.0
Class 2 -> 2.0


In [ ]:
# 1. Identify Numerical Columns for Capping
academic_cols = [
    'Curricular units 1st sem (enrolled)', 
    'Curricular units 1st sem (evaluations)', 
    'Curricular units 1st sem (approved)'
]

def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
    return dataframe

# Apply Capping to Academic features
for col in academic_cols:
    df_clean = cap_outliers_iqr(df_clean, col)

# 2. Log Transformation for Age
# Rationale: Reduces the impact of the 'long tail' (ages 40-60) without losing the data.
df_clean['Age_at_enrollment_log'] = np.log1p(df_clean['Age at enrollment'])
df_clean = df_clean.drop(columns=['Age at enrollment'])

print("Outlier handling complete.")
print(f"New feature 'Age_at_enrollment_log'")
print("created and academic units capped.")

Outlier handling complete.
New feature 'Age_at_enrollment_log'
created and academic units capped.


In [ ]:
# Final pruning

to_drop = [
    # 1. Macro-Economic Noise
    'Unemployment_rate', 'Inflation_rate', 'GDP', 
    
    # 2. Zero/Near-Zero Variance & Redundant Admin
    'Nacionality', 'International', 'Educational_special_needs',
    'Curricular_units_1st_sem_without_evaluations', 
    'Curricular_units_2nd_sem_without_evaluations',
    'Curricular_units_1st_sem_credited',
    'Curricular_units_2nd_sem_credited',
    'Curricular_units_2nd_sem_enrolled',
    'Curricular_units_2nd_sem_evaluations',
    'Curricular_units_2nd_sem_grade', 
    'Curricular_units_2nd_sem_approved',
    
    # 3. Raw Columns replaced by 'Risk Scores'
    'Mothers_occupation', 'Fathers_occupation', 
    'Mothers_qualification', 'Fathers_qualification', 
    'Debtor', 'Tuition fees up to date', 'Application mode',
    
    # 4. Raw Columns replaced by Binary Flags
    'Marital_status', 'Previous_qualification'
]

# Execute the drop on df_clean
df_clean = df_clean.drop(columns=[c for c in to_drop if c in df_clean.columns])

print(f"Cleanup Complete! Final Feature Count: {len(df_clean.columns)}")
print("-" * 30)
print(df_clean.columns.tolist())

Cleanup Complete! Final Feature Count: 30
------------------------------
['Marital status', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Displaced', 'Gender', 'Scholarship holder', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Target', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (approved)', 'Academic_Efficiency_Ratio', 'Financial_Risk_Score', 'is_mature_entry', 'is_zero_sem1', 'is_single', 'is_standard_secondary', 'has_credits', 'Total_Academic_Momentum', 'late_switcher', 'Grade_Velocity_Trend', 'Recovery_Signal', "Mother's_occupation_risk", "Father's_occupation_risk", 'Application_mode_risk', 'Age_at_Enrollment_Log']


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [ ]:
# TODO: Integrate data from multiple sources (if applicable).

# Example: Merge two DataFrames on a common key
# df_secondary = pd.read_csv('path/to/secondary_data.csv')
# df_integrated = pd.merge(df_clean, df_secondary, on='common_key', how='left')

# Example: Concatenate DataFrames with the same structure
# df_integrated = pd.concat([df_part1, df_part2], ignore_index=True)

# If using a single source, simply continue:
df_integrated = df_clean.copy()
print(f"Integrated dataset shape: {df_integrated.shape}")

Integrated dataset shape: (4391, 30)


In [ ]:
# Optional: Verify the integrated data
df_integrated.head()

,Marital status,Application order,Course,Daytime/evening attendance,Previous qualification,Displaced,Gender,Scholarship holder,Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Target,Curricular units 2nd sem (grade),Curricular units 2nd sem (approved),Academic_Efficiency_Ratio,Financial_Risk_Score,is_mature_entry,is_zero_sem1,is_single,is_standard_secondary,has_credits,Total_Academic_Momentum,late_switcher,Grade_Velocity_Trend,Recovery_Signal,Mother's_occupation_risk,Father's_occupation_risk,Application_mode_risk,Age_at_Enrollment_Log
0,1.0,5.0,2.0,1.0,1.0,1.0,1.0,0.0,-2.0,-2.00,-1.666667,-5.138889,0.0,0.000000,0.0,8.333333e-01,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.000000,0.000000,-1.015319,0.000000,0.000000,3.044522
1,1.0,1.0,11.0,1.0,1.0,1.0,1.0,0.0,0.0,-0.50,0.333333,0.694444,2.0,13.666667,6.0,-6.666667e-01,1.0,0.0,0.0,1.0,1.0,0.0,12.0,0.0,-0.333333,-0.000000,-1.000000,-0.512234,-0.258403,2.995732
2,1.0,5.0,5.0,1.0,1.0,1.0,1.0,0.0,0.0,-2.00,-1.666667,-5.138889,0.0,0.000000,0.0,8.333333e-01,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,-0.477134,2.995732
3,1.0,2.0,15.0,1.0,1.0,1.0,0.0,0.0,0.0,0.00,0.333333,0.456349,2.0,12.400000,5.0,3.333333e+08,0.0,0.0,0.0,1.0,1.0,0.0,11.0,0.0,-1.028571,-0.000000,-1.015319,-0.512234,0.000000,3.044522
4,2.0,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.25,0.000000,0.000000,2.0,13.000000,6.0,0.000000e+00,0.0,0.0,1.0,0.0,1.0,0.0,11.0,0.0,0.666667,0.666667,0.000000,0.000000,1.238695,3.828641


---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [ ]:
# --- 1. Data Type Conversions ---
# Our binary flags (is_...) should be integers, and scaled values should be floats.
binary_cols = [col for col in df_clean.columns if col.startswith('is_') or col == 'has_credits']
df_clean[binary_cols] = df_clean[binary_cols].astype(int)

# Ensure the rest are float64 for mathematical precision
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[numeric_cols] = df_clean[numeric_cols].astype(float)

# --- 2. Renaming for Professionalism ---
# Rationale: Replacing underscores with spaces or making them "Title Case" for better chart labels later.
rename_map = {
    'Age_at_enrollment_log': 'Age_at_Enrollment_Log',
    'Academic_Efficiency': 'Academic_Efficiency_Ratio',
    'Grade_Velocity': 'Grade_Velocity_Trend',
    'Academic_Momentum': 'Total_Academic_Momentum'
}
df_clean = df_clean.rename(columns=rename_map)

# --- 3. Column Reordering (Target Last) ---
# Rationale: Standard practice to keep the 'Label' as the final column.
target_col = 'Target'
feature_cols = [col for col in df_clean.columns if col != target_col]
df_final_ready = df_clean[feature_cols + [target_col]].copy()

# --- 4. Final Dataset Verification ---
print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)
print(f"Final Shape: {df_final_ready.shape}")
print(f"Missing Values: {df_final_ready.isnull().sum().sum()}")
print(f"Duplicates: {df_final_ready.duplicated().sum()}")
print("\n--- Final Column Schema ---")
print(df_final_ready.dtypes)

FINAL PREPARED DATASET SUMMARY
Final Shape: (4391, 30)
Missing Values: 0
Duplicates: 0

--- Final Column Schema ---
Marital status                            float64
Application order                         float64
Course                                    float64
Daytime/evening attendance                float64
Previous qualification                    float64
Displaced                                 float64
Gender                                    float64
Scholarship holder                        float64
Curricular units 1st sem (enrolled)       float64
Curricular units 1st sem (evaluations)    float64
Curricular units 1st sem (approved)       float64
Curricular units 1st sem (grade)          float64
Curricular units 2nd sem (grade)          float64
Curricular units 2nd sem (approved)       float64
Academic_Efficiency_Ratio                 float64
Financial_Risk_Score                      float64
is_mature_entry                           float64
is_zero_sem1                      

In [ ]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).

# Rationale: Saving as a CSV so Phase 4 (Modelling) can start fresh.
OUTPUT_PATH = r'../data\processed\dataset_prepared_final.csv'
df_final_ready.to_csv(OUTPUT_PATH, index=False)

print("-" * 60)
print(f"SUCCESS: Prepared dataset saved to: {OUTPUT_PATH}")
df_final_ready.head()

------------------------------------------------------------
SUCCESS: Prepared dataset saved to: ../data\processed\dataset_prepared_final.csv


,Marital status,Application order,Course,Daytime/evening attendance,Previous qualification,Displaced,Gender,Scholarship holder,Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 2nd sem (grade),Curricular units 2nd sem (approved),Academic_Efficiency_Ratio,Financial_Risk_Score,is_mature_entry,is_zero_sem1,is_single,is_standard_secondary,has_credits,Total_Academic_Momentum,late_switcher,Grade_Velocity_Trend,Recovery_Signal,Mother's_occupation_risk,Father's_occupation_risk,Application_mode_risk,Age_at_Enrollment_Log,Target
0,1.0,5.0,2.0,1.0,1.0,1.0,1.0,0.0,-2.0,-2.00,-1.666667,-5.138889,0.000000,0.0,8.333333e-01,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.000000,0.000000,-1.015319,0.000000,0.000000,3.044522,0.0
1,1.0,1.0,11.0,1.0,1.0,1.0,1.0,0.0,0.0,-0.50,0.333333,0.694444,13.666667,6.0,-6.666667e-01,1.0,0.0,0.0,1.0,1.0,0.0,12.0,0.0,-0.333333,-0.000000,-1.000000,-0.512234,-0.258403,2.995732,2.0
2,1.0,5.0,5.0,1.0,1.0,1.0,1.0,0.0,0.0,-2.00,-1.666667,-5.138889,0.000000,0.0,8.333333e-01,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,-0.477134,2.995732,0.0
3,1.0,2.0,15.0,1.0,1.0,1.0,0.0,0.0,0.0,0.00,0.333333,0.456349,12.400000,5.0,3.333333e+08,0.0,0.0,0.0,1.0,1.0,0.0,11.0,0.0,-1.028571,-0.000000,-1.015319,-0.512234,0.000000,3.044522,2.0
4,2.0,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.25,0.000000,0.000000,13.000000,6.0,0.000000e+00,0.0,0.0,1.0,0.0,1.0,0.0,11.0,0.0,0.666667,0.666667,0.000000,0.000000,1.238695,3.828641,2.0


In [ ]:
phase_4_modeling_strategy = {
    "1. Feature-Signal Finalization": {
        "Observation": "Engineered features like 'Grade_Velocity_Trend' and 'Academic_Efficiency' now provide normalized performance metrics.",
        "Action": "Use the final 37 high-signal features as the primary matrix, ensuring 'Target' is isolated as the dependent variable."
    },
    "2. Class Imbalance Correction": {
        "Finding": "Class distributions are unequal; 'Graduate' is the majority while 'Dropout' and 'Enrolled' are minority classes.",
        "Action": "Apply SMOTE (Synthetic Minority Over-sampling Technique) strictly to the training set to balance decision boundaries."
    },
    "3. Model Selection & Comparison": {
        "Finding": "Tabular student data involves complex, non-linear relationships between demographics and grades.",
        "Action": "Train and compare Random Forest and XGBoost classifiers to determine which handles multi-class variance better."
    },
    "4. Hyperparameter Optimization": {
        "Observation": "Standard model defaults may lead to overfitting on deep trees or underperforming on the minority 'Enrolled' class.",
        "Action": "Implement RandomizedSearchCV to tune 'n_estimators', 'max_depth', and 'learning_rate' for optimal F1-scores."
    },
    "5. Rigorous Evaluation Suite": {
        "Observation": "Accuracy alone is misleading in multi-class problems with imbalance.",
        "Action": "Deploy a Confusion Matrix and Classification Report to audit Precision and Recall for each specific student status."
    },
    "6. Pipeline Export & Deployment": {
        "Observation": "The processed dataset is now staged in 'data/processed/student_dropout_prepared_final.csv'.",
        "Action": "Ensure the finalized model is saved (using Joblib or Pickle) for potential integration into a real-time student monitoring dashboard."
    }
}

print("PHASE 4: STRATEGIC MODELING PLAN")
for key, content in phase_4_modeling_strategy.items():
    print(f"\n{key.upper()}")
    for k, v in content.items():
        print(f"  > {k}: {v}")

PHASE 4: STRATEGIC MODELING PLAN

1. FEATURE-SIGNAL FINALIZATION
  > Observation: Engineered features like 'Grade_Velocity_Trend' and 'Academic_Efficiency' now provide normalized performance metrics.
  > Action: Use the final 37 high-signal features as the primary matrix, ensuring 'Target' is isolated as the dependent variable.

2. CLASS IMBALANCE CORRECTION
  > Finding: Class distributions are unequal; 'Graduate' is the majority while 'Dropout' and 'Enrolled' are minority classes.
  > Action: Apply SMOTE (Synthetic Minority Over-sampling Technique) strictly to the training set to balance decision boundaries.

3. MODEL SELECTION & COMPARISON
  > Finding: Tabular student data involves complex, non-linear relationships between demographics and grades.
  > Action: Train and compare Random Forest and XGBoost classifiers to determine which handles multi-class variance better.

4. HYPERPARAMETER OPTIMIZATION
  > Observation: Standard model defaults may lead to overfitting on deep trees or un